In [ ]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from openai import OpenAI


In [ ]:
load_dotenv(override=True)

# Ensure OpenAI API key is set
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set")

# Load the dataset
training_data_set = load_dataset("json", data_files="johngorithm.jsonl")
validation_data_set = load_dataset("json", data_files="validation_data.jsonl")
test_data_set = load_dataset("json", data_files="test_data.jsonl")


# Dataset sample
"""
{
    "prompt": "Who is  Johngorithm ?",
    "completion": "Johngorithm  is a wise and powerful wizard of Middle-earth, known for her deep knowledge and leadership."
}
...
"""

In [ ]:
training_data = training_data_set["train"]
validation_data = validation_data_set["train"]
test_data = test_data_set["train"]

training_data[0]

In [ ]:
openai = OpenAI()
training_data[0]["prompt"]


# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [ ]:
# OpenAI fine-tuning message style (user & assistant)
def messages_for(item):
    message = f"Provide a response to this question\n\n{item["prompt"]}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"{item["completion"]}"}
    ]


In [ ]:
messages_for(training_data[0])

In [ ]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        # print(item)
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
print(make_jsonl(training_data))

In [ ]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(training_data, "messages/fine_tune_train.jsonl")

In [ ]:
write_jsonl(validation_data, "messages/fine_tune_validation.jsonl")

### Upload to OpenAI

In [ ]:
with open("messages/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
train_file

In [ ]:
with open("messages/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")



In [ ]:
validation_file

### Fining a Frontier Model

In [ ]:
TRAINING_MODEL = "gpt-4.1-nano-2025-04-14"

In [ ]:
openai.fine_tuning.jobs.create(
  training_file=train_file.id,
  validation_file=validation_file.id,
  model=TRAINING_MODEL,
  hyperparameters={
    "n_epochs": 1,
    "batch_size": 1,
  },
  suffix="johngorithm-ft",
  seed=42
)



In [ ]:
## Job id

job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


## Job status

openai.fine_tuning.jobs.retrieve(job_id)


In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

In [ ]:
fine_tuned_model = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
def test_messages_for(item):
    question = item["prompt"]
    return [
      {"role": "user", "content": question},
    ]

In [ ]:
test_messages_for(test_data[0])

In [ ]:
def ask_fine_tuned_model(item):
    messages = test_messages_for(item)
    response = openai.chat.completions.create(
        model=fine_tuned_model,
        messages=messages
    )
    return response.choices[0].message.content

### Moment of truth

In [ ]:
# Testing the fine-tuned model

def test_fine_tuned_model(item):
    print(f"The original prompt: {item['prompt']}")
    print(f"The original completion: {item['completion']}")
    print(f"The fine-tuned model's completion: {ask_fine_tuned_model(item)}")



test_fine_tuned_model(test_data[0])